# Tutorial: Multi-Hop RAG for Question Answering

This tutorial walks you through building a **multi-hop retrieval-augmented generation (RAG)** system in DSPy. Multi-hop RAG is essential when answering complex questions that cannot be resolved with a single retrieval step—for example, _"Who was the director of the film featuring the actor who starred in Inception?"_ requires first finding the actor, then finding the director of a different film they appeared in.

We'll build a 2-hop pipeline that:
1. Generates an initial search query from the question.
2. Retrieves relevant passages.
3. Generates a follow-up query using what was learned.
4. Retrieves again with the refined query.
5. Synthesizes a final answer from all gathered context.

Install the latest DSPy via `pip install -U dspy` and follow along. You also need to run `pip install datasets`.

**See also:**
- [Basic RAG tutorial](../rag/index.ipynb) — single-hop RAG for Q&A.
- [Multi-Hop Retrieval tutorial](../multihop_search/index.ipynb) — multi-hop search for claim verification.

<details>
<summary>Recommended: Set up MLflow Tracing to understand what's happening under the hood.</summary>

### MLflow DSPy Integration

<a href="https://mlflow.org/">MLflow</a> is an LLMOps tool that natively integrates with DSPy and offers explainability and experiment tracking. In this tutorial, you can use MLflow to visualize prompts and optimization progress as traces to understand DSPy's behavior better. You can set it up by following the four steps below.

1. Install MLflow

```bash
%pip install mlflow>=2.20
```

2. Start MLflow UI in a separate terminal
```bash
mlflow ui --port 5000
```

3. Connect the notebook to MLflow
```python
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("DSPy")
```

4. Enable tracing.
```python
mlflow.dspy.autolog()
```

To learn more about the integration, visit [MLflow DSPy Documentation](https://mlflow.org/docs/latest/llms/dspy/index.html).
</details>

## Configure DSPy

We'll use OpenAI's `gpt-4o-mini` as our language model. DSPy looks for `OPENAI_API_KEY` in your environment. You can swap this out for [other providers or local models](https://github.com/stanfordnlp/dspy/blob/main/examples/migration.ipynb).

In [ ]:
import dspy

lm = dspy.LM('openai/gpt-4o-mini')
dspy.configure(lm=lm)

## Set Up the Retriever

We'll use a public ColBERTv2 server that indexes Wikipedia 2017 abstracts. This lets us do semantic retrieval over millions of passages without any local setup.

In [ ]:
retriever = dspy.ColBERTv2(url='http://20.102.90.50:2017/wiki17_abstracts')

def search(query: str, k: int = 3) -> list[str]:
    """Retrieve top-k passages from Wikipedia using ColBERTv2."""
    results = retriever(query, k=k)
    return [x['text'] for x in results]

Let's do a quick sanity check:

In [ ]:
passages = search("Christopher Nolan director films")
for p in passages:
    print(p[:120])
    print()

## Load the HotPotQA Dataset

[HotPotQA](https://hotpotqa.github.io/) is a multi-hop question answering dataset where each question requires reasoning across at least two Wikipedia articles. It's a natural fit for evaluating multi-hop RAG.

We'll load it and create `dspy.Example` objects, specifying `question` as the input field.

In [ ]:
import random
from dspy.datasets import HotPotQA

dataset = HotPotQA(train_seed=1, train_size=200, eval_seed=2025, dev_size=300, test_size=0)

trainset = [x.with_inputs('question') for x in dataset.train]
devset   = [x.with_inputs('question') for x in dataset.dev]

print(f"Train: {len(trainset)} | Dev: {len(devset)}")

In [ ]:
# Inspect a sample
example = trainset[0]
print("Question:", example.question)
print("Answer:", example.answer)

## Define Signatures

DSPy [Signatures](https://dspy.ai/learn/programming/signatures) are typed input/output schemas that tell DSPy modules what to do. We need two:

1. **`GenerateSearchQuery`** — Given a question and any context gathered so far, produce a focused search query for the next retrieval hop.
2. **`GenerateAnswer`** — Given all retrieved context and the original question, produce the final answer.

In [ ]:
class GenerateSearchQuery(dspy.Signature):
    """Given a question and context gathered so far, generate a focused search query to retrieve the next piece of evidence."""

    context: list[str] = dspy.InputField(desc="Passages retrieved in previous hops (empty on the first hop)")
    question: str = dspy.InputField()
    query: str = dspy.OutputField(desc="A concise search query targeting the missing piece of information")


class GenerateAnswer(dspy.Signature):
    """Answer a question using retrieved context. Be concise — a word or short phrase is usually enough."""

    context: list[str] = dspy.InputField(desc="Retrieved passages from all hops")
    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="A concise factual answer")

## Build the Multi-Hop RAG Module

We compose our signatures and `dspy.ChainOfThought` into a `dspy.Module`. The module runs `num_hops` retrieval steps—each one generating a query, retrieving passages, and accumulating context—then produces a final answer.

`dspy.ChainOfThought` wraps a signature and automatically elicits step-by-step reasoning before the model commits to its output, which is especially helpful when queries need to build on prior evidence.

In [ ]:
class MultiHopRAG(dspy.Module):
    def __init__(self, num_hops: int = 2, passages_per_hop: int = 3):
        self.num_hops = num_hops
        self.passages_per_hop = passages_per_hop

        # ChainOfThought lets the LM reason before generating each query.
        self.generate_query = dspy.ChainOfThought(GenerateSearchQuery)
        self.generate_answer = dspy.ChainOfThought(GenerateAnswer)

    def forward(self, question: str) -> dspy.Prediction:
        context = []

        for _ in range(self.num_hops):
            # Generate a targeted query based on accumulated context
            query = self.generate_query(context=context, question=question).query
            # Retrieve new passages and extend the context window
            new_passages = search(query, k=self.passages_per_hop)
            context += new_passages

        # Synthesize a final answer from all gathered context
        prediction = self.generate_answer(context=context, question=question)
        return dspy.Prediction(context=context, answer=prediction.answer)

## Try It Out

Let's run the off-the-shelf (zero-shot) module on a HotPotQA question to see what it produces.

In [ ]:
multihop_rag = MultiHopRAG(num_hops=2, passages_per_hop=3)

question = "Who was the director of the 1972 film that won the Academy Award for Best Picture?"
pred = multihop_rag(question=question)

print("Question:", question)
print("Answer:", pred.answer)
print("\n--- Retrieved context (first 150 chars each) ---")
for i, passage in enumerate(pred.context, 1):
    print(f"[{i}] {passage[:150]}")

In [ ]:
# Inspect the prompts sent by DSPy in both hops
dspy.inspect_history(n=3)

## Evaluate the Baseline

We'll use DSPy's built-in `answer_exact_match` metric, which checks whether the predicted answer contains the gold answer (case-insensitive). For production systems you'd want something richer like SemanticF1, but exact match is a clear baseline for factual Q&A.

In [ ]:
from dspy.evaluate import answer_exact_match

evaluate = dspy.Evaluate(
    devset=devset,
    metric=answer_exact_match,
    num_threads=16,
    display_progress=True,
    display_table=3,
)

baseline_score = evaluate(MultiHopRAG())
print(f"\nBaseline exact match: {baseline_score:.1f}%")

## Optimize with BootstrapFewShot

DSPy optimizers can automatically find few-shot examples that improve performance. `BootstrapFewShot` runs the program on training examples, filters for ones where the metric passes, and adds them as demonstrations to the prompts.

This is the simplest optimizer—a great first step before reaching for the more powerful (and more expensive) `MIPROv2`.

In [ ]:
from dspy.teleprompt import BootstrapFewShot

optimizer = BootstrapFewShot(metric=answer_exact_match, max_bootstrapped_demos=3)
optimized_rag = optimizer.compile(MultiHopRAG(), trainset=trainset)

In [ ]:
optimized_score = evaluate(optimized_rag)
print(f"\nOptimized exact match: {optimized_score:.1f}%")
print(f"Improvement: +{optimized_score - baseline_score:.1f}%")

## Optional: Optimize with MIPROv2

For larger gains, `MIPROv2` jointly optimizes instructions _and_ demonstrations across all sub-modules. It's more expensive (makes more LM calls) but often yields substantially better prompts.

```python
tp = dspy.MIPROv2(metric=answer_exact_match, auto="medium", num_threads=16)

mipro_rag = tp.compile(
    MultiHopRAG(),
    trainset=trainset,
    max_bootstrapped_demos=3,
    max_labeled_demos=3,
)

evaluate(mipro_rag)
```

## Save and Reload

Once you're happy with the optimized program, save it to disk so you can reuse it without re-running optimization.

In [ ]:
optimized_rag.save("optimized_multihop_rag.json")

# Reload later
loaded_rag = MultiHopRAG()
loaded_rag.load("optimized_multihop_rag.json")

pred = loaded_rag(question="Who was the director of the 1972 film that won the Academy Award for Best Picture?")
print("Answer:", pred.answer)

## Summary

Here's what we built:

| Component | Role |
|---|---|
| `GenerateSearchQuery` | Signature: turns context + question into a retrieval query |
| `GenerateAnswer` | Signature: synthesizes final answer from all passages |
| `dspy.ChainOfThought` | Module: adds reasoning before each output |
| `MultiHopRAG` | Program: orchestrates N hops of retrieve-then-query |
| `BootstrapFewShot` | Optimizer: auto-selects few-shot demos from training data |

The key insight is that **the same optimizer API works regardless of how complex your program is**. Whether you have 1 sub-module or 10, DSPy optimizes them jointly.

## What's Next?

- **More hops**: Try `num_hops=3` for questions that require longer reasoning chains.
- **Better retrieval**: Swap ColBERTv2 for a local BM25 index (see [Multi-Hop Retrieval tutorial](../multihop_search/index.ipynb)) or dense embeddings (see [RAG tutorial](../rag/index.ipynb)).
- **Stronger optimization**: Replace `BootstrapFewShot` with `MIPROv2` for instruction-level optimization.
- **Agent-style**: Instead of a fixed number of hops, try [Building RAG as Agent](../agents/index.ipynb) for adaptive retrieval.